# SNP Selection — AA CPD (DoubleML + Stability Selection, Fully Corrected)

**Purpose:** Identify SNPs causally associated with CPD (smokers only) via DoubleML +
stability selection, on relatedness-filtered data with MAF floor applied within the
smoker subsample (see prior investigation: median MAF 0.06% in uncorrected smoker
subset caused PC-algorithm MemoryError).

**Inputs:** `checkpoint7b_snp_encoded_012_relatedness_filtered.csv`,
`checkpoint2b_metadata_relatedness_filtered.csv`.

**Method:** Subset to smokers → MAF floor (≥1%) within that subsample → EIGENSTRAT
standardize → PCA (full SVD) within smoker subsample specifically (not reusing the
full-cohort confounders, since the analysis subsample differs) → DoubleML stability
selection.

**Threshold policy:** Given today's finding that intermediate stability thresholds
(80-90%) can produce PC-algorithm-intractable networks even after cleaning, this
notebook goes directly to 100% stability as the primary/only network for CPD,
skipping exploratory intermediate thresholds. Lower-stability results (if computed)
are reserved for pathway enrichment only, not causal graph construction.

## Step 1 — Subset to smokers, apply MAF floor

In [1]:
import pandas as pd
import numpy as np
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

meta_df = pd.read_csv(os.path.join(out_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
smokers = meta_df[meta_df["smoking_status"] == "Smoker"].copy()
smokers["cpd"] = pd.to_numeric(smokers["cpd"], errors="coerce")
smokers = smokers[smokers["cpd"] >= 0].dropna(subset=["cpd"])
print("Smoker sample count (relatedness-filtered):", len(smokers))

encoded_df = pd.read_csv(os.path.join(out_dir, "checkpoint7b_snp_encoded_012_relatedness_filtered.csv"))
probe_id_array = encoded_df["probe_id"].to_numpy()
all_sample_ids = encoded_df.columns[1:].tolist()
smoker_id_set = set(smokers["sample_id"].tolist())
smoker_cols = [s for s in all_sample_ids if s in smoker_id_set]
print("Smoker columns matched in genotype file:", len(smoker_cols))

X_snp_smokers = encoded_df.loc[:, smoker_cols].to_numpy(dtype=np.float64).T
print("X_snp_smokers shape (smokers x all SNPs):", X_snp_smokers.shape)

p_smoker = X_snp_smokers.mean(axis=0) / 2
maf_smoker = np.minimum(p_smoker, 1 - p_smoker)
maf_ok_mask = maf_smoker >= 0.01
print("SNPs passing MAF >= 1%:", maf_ok_mask.sum(), "/", len(maf_ok_mask))

Smoker sample count (relatedness-filtered): 1459
Smoker columns matched in genotype file: 1459
X_snp_smokers shape (smokers x all SNPs): (1459, 238927)
SNPs passing MAF >= 1%: 49810 / 238927


## Step 2 — Exclude sex-linked probes, standardize (native float64), build PCA confounders

In [2]:
import pandas as pd
import numpy as np
import re
import os
import gc

manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_probes = set(manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "IlmnID"])
sex_linked_core_names = {strip_address_suffix(p) for p in sex_linked_probes}
our_probe_core_names = {strip_address_suffix(p): p for p in probe_id_array}
to_exclude = {our_probe_core_names[c] for c in sex_linked_core_names if c in our_probe_core_names}
del manifest_df
gc.collect()

# combine sex-linked exclusion with the MAF floor from Step 1
sex_linked_mask = ~np.isin(probe_id_array, list(to_exclude))
combined_mask = sex_linked_mask & maf_ok_mask
print("SNPs after sex-linked exclusion + MAF floor:", combined_mask.sum())

X_snp_final = X_snp_smokers[:, combined_mask]
probe_ids_final = probe_id_array[combined_mask]
del X_snp_smokers
gc.collect()

p_final = X_snp_final.mean(axis=0) / 2
denom_final = np.sqrt(2 * p_final * (1 - p_final))
X_standardized_cpd = (X_snp_final - 2 * p_final) / denom_final
print("Standardized CPD matrix shape:", X_standardized_cpd.shape)

SNPs after sex-linked exclusion + MAF floor: 48624
Standardized CPD matrix shape: (1459, 48624)


## Step 3 — PCA (smoker-subsample specific) + confounder assembly

In [3]:
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

pca_cpd = PCA(n_components=10, random_state=42, svd_solver='full')
pcs_cpd = pca_cpd.fit_transform(X_standardized_cpd)

print("Explained variance ratio:", pca_cpd.explained_variance_ratio_)
print("Cumulative:", np.cumsum(pca_cpd.explained_variance_ratio_))

meta_smoker_aligned = smokers.set_index("sample_id").reindex(smoker_cols).reset_index()
age_std_cpd = ((meta_smoker_aligned["age"].astype(float) - meta_smoker_aligned["age"].astype(float).mean()) /
               meta_smoker_aligned["age"].astype(float).std()).values
gender_binary_cpd = (meta_smoker_aligned["gender"] == "Male").astype(np.float64).values

X_confounders_cpd = np.hstack([
    pcs_cpd,
    age_std_cpd.reshape(-1, 1),
    gender_binary_cpd.reshape(-1, 1)
]).astype(np.float64)

print("Confounders shape:", X_confounders_cpd.shape)

np.save(os.path.join(out_dir, "confounders_X_cpd.npy"), X_confounders_cpd)
print("Saved.")

Y_cpd = smokers.set_index("sample_id").reindex(smoker_cols)["cpd"].values.astype(np.float64)
print("Y_cpd shape:", Y_cpd.shape, "range:", Y_cpd.min(), "-", Y_cpd.max())

Explained variance ratio: [0.01693242 0.00245273 0.00230566 0.00202099 0.00180912 0.00179771
 0.00171667 0.00162834 0.00157491 0.00153161]
Cumulative: [0.01693242 0.01938515 0.02169081 0.0237118  0.02552092 0.02731863
 0.0290353  0.03066364 0.03223855 0.03377015]
Confounders shape: (1459, 12)
Saved.
Y_cpd shape: (1459,) range: 1.0 - 60.0


## Step 4 — VALIDATION GATE: Genomic inflation factor before proceeding

In [4]:
import numpy as np
from sklearn.model_selection import KFold
from scipy import stats
import time

def doubleml_scan(X_snps, Y, X_conf, n_folds=5, random_state=42):
    n, n_snps = X_snps.shape
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    D_resid = np.zeros_like(X_snps)
    Y_resid = np.zeros(n)
    Xc = np.column_stack([np.ones(n), X_conf])

    for train_idx, test_idx in kf.split(Xc):
        Xc_tr, Xc_te = Xc[train_idx], Xc[test_idx]
        coef_Y = np.linalg.lstsq(Xc_tr, Y[train_idx], rcond=None)[0]
        Y_resid[test_idx] = Y[test_idx] - Xc_te @ coef_Y
        coef_D = np.linalg.lstsq(Xc_tr, X_snps[train_idx], rcond=None)[0]
        D_resid[test_idx] = X_snps[test_idx] - Xc_te @ coef_D

    Yr = Y_resid - Y_resid.mean()
    Dr = D_resid - D_resid.mean(axis=0)
    del D_resid

    sum_DY = (Dr * Yr[:, None]).sum(axis=0)
    sum_DD = (Dr ** 2).sum(axis=0)
    sum_YY = (Yr ** 2).sum()

    theta = sum_DY / sum_DD
    ssr = sum_YY - (sum_DY ** 2) / sum_DD
    sigma2 = ssr / (n - 2)
    se = np.sqrt(sigma2 / sum_DD)
    t_stat = theta / se
    pvals = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 2))
    return theta, pvals

def compute_lambda(pvals):
    chi2_observed = stats.chi2.isf(pvals, df=1)
    return np.median(chi2_observed) / stats.chi2.ppf(0.5, df=1)

print("Running 5 seeds for stable lambda estimate...")
lambdas = []
for seed in range(5):
    _, pv = doubleml_scan(X_standardized_cpd, Y_cpd, X_confounders_cpd, random_state=seed)
    lam = compute_lambda(pv)
    lambdas.append(lam)
    print(f"seed={seed}: lambda={lam:.4f}")

mean_lambda = np.mean(lambdas)
print(f"\nMean lambda: {mean_lambda:.4f} (+/- {np.std(lambdas):.4f})")

Running 5 seeds for stable lambda estimate...
seed=0: lambda=0.9976
seed=1: lambda=0.9907
seed=2: lambda=0.9868
seed=3: lambda=0.9941
seed=4: lambda=0.9814

Mean lambda: 0.9901 (+/- 0.0056)


In [5]:
n_repeats = 30
threshold = 0.0001
n_snps = X_standardized_cpd.shape[1]
significant_counts_cpd = np.zeros(n_snps, dtype=int)

start = time.time()
for rep in range(n_repeats):
    theta_rep, pval_rep = doubleml_scan(X_standardized_cpd, Y_cpd, X_confounders_cpd, random_state=rep)
    significant_counts_cpd += (pval_rep < threshold).astype(int)
    if (rep + 1) % 5 == 0:
        print(f"Completed {rep+1}/{n_repeats}, elapsed {time.time()-start:.1f}s")

print(f"Total time: {time.time()-start:.1f}s")
stability_fraction_cpd = significant_counts_cpd / n_repeats
print("\nStability distribution:")
for t in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    print(f"  >= {t:.0%}: {(stability_fraction_cpd >= t).sum()} SNPs")

Completed 5/30, elapsed 31.8s
Completed 10/30, elapsed 63.1s
Completed 15/30, elapsed 94.3s
Completed 20/30, elapsed 125.3s
Completed 25/30, elapsed 156.5s
Completed 30/30, elapsed 187.5s
Total time: 187.5s

Stability distribution:
  >= 50%: 5 SNPs
  >= 60%: 5 SNPs
  >= 70%: 5 SNPs
  >= 80%: 4 SNPs
  >= 90%: 4 SNPs
  >= 100%: 3 SNPs


In [6]:
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

stability_df_cpd = pd.DataFrame({
    "probe_id": probe_ids_final,
    "stability_fraction": stability_fraction_cpd,
    "n_significant_repeats": significant_counts_cpd
})
stability_df_cpd = stability_df_cpd.sort_values("stability_fraction", ascending=False)

top5 = stability_df_cpd[stability_df_cpd["stability_fraction"] >= 0.5]
print(top5)

ppp1r12b_check = top5[top5["probe_id"].str.contains("2277017")]
print("\nPPP1R12B present:", len(ppp1r12b_check) > 0)

stability_df_cpd.to_csv(os.path.join(out_dir, "checkpoint9_doubleml_stability_results_cpd.csv"), index=False)
print("Saved full results.")

                          probe_id  stability_fraction  n_significant_repeats
13128  exm2265806-0_B_R_1984855512            1.000000                     30
3742   exm2277017-0_T_R_1989215336            1.000000                     30
9002    exm318398-0_B_F_1922126681            1.000000                     30
8373    exm294480-0_B_F_1922170143            0.933333                     28
5985    exm209142-0_B_F_1919018703            0.766667                     23

PPP1R12B present: True
Saved full results.


In [7]:
import pandas as pd
import re
import numpy as np

manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_address_suffix)
position_lookup_cpd = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]

shortlist_cpd = stability_df_cpd[stability_df_cpd["stability_fraction"] >= 0.8].copy()
shortlist_cpd["core_name"] = shortlist_cpd["probe_id"].map(strip_address_suffix)
shortlist_cpd_pos = shortlist_cpd.merge(position_lookup_cpd, left_on="core_name", right_index=True, how="left")
print(shortlist_cpd_pos[["probe_id", "Chr", "MapInfo", "stability_fraction"]])

probe_id_to_idx_cpd = {pid: i for i, pid in enumerate(probe_ids_final)}
def get_geno_cpd(pid):
    return X_standardized_cpd[:, probe_id_to_idx_cpd[pid]]

X_check_cpd = np.column_stack([get_geno_cpd(pid) for pid in shortlist_cpd_pos["probe_id"]])
corr_cpd = np.corrcoef(X_check_cpd.T)
print("\nPairwise correlations:")
print(pd.DataFrame(corr_cpd, index=shortlist_cpd_pos["probe_id"].str[:20], columns=shortlist_cpd_pos["probe_id"].str[:20]))

                          probe_id Chr      MapInfo  stability_fraction
13128  exm2265806-0_B_R_1984855512   4  183777997.0            1.000000
3742   exm2277017-0_T_R_1989215336   1  202399880.0            1.000000
9002    exm318398-0_B_F_1922126681   3   50614990.0            1.000000
8373    exm294480-0_B_F_1922170143   3   16419576.0            0.933333

Pairwise correlations:
probe_id              exm2265806-0_B_R_198  exm2277017-0_T_R_198  \
probe_id                                                           
exm2265806-0_B_R_198              1.000000             -0.067994   
exm2277017-0_T_R_198             -0.067994              1.000000   
exm318398-0_B_F_1922              0.009417              0.008301   
exm294480-0_B_F_1922             -0.064811              0.974078   

probe_id              exm318398-0_B_F_1922  exm294480-0_B_F_1922  
probe_id                                                          
exm2265806-0_B_R_198              0.009417             -0.064811  
exm227

In [8]:
final_shortlist_cpd = shortlist_cpd_pos[shortlist_cpd_pos["probe_id"] != "exm294480-0_B_F_1922170143"].copy()
print("Final CPD shortlist:", len(final_shortlist_cpd))
print(final_shortlist_cpd[["probe_id", "Chr", "MapInfo", "stability_fraction"]])

import os
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
final_shortlist_cpd.to_csv(os.path.join(out_dir, "shortlist_cpd_final.csv"), index=False)
print("Saved.")

Final CPD shortlist: 3
                          probe_id Chr      MapInfo  stability_fraction
13128  exm2265806-0_B_R_1984855512   4  183777997.0                 1.0
3742   exm2277017-0_T_R_1989215336   1  202399880.0                 1.0
9002    exm318398-0_B_F_1922126681   3   50614990.0                 1.0
Saved.


In [9]:
import numpy as np
import json
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

final_ids_cpd = final_shortlist_cpd["probe_id"].tolist()
final_col_indices_cpd = [probe_id_to_idx_cpd[pid] for pid in final_ids_cpd]
X_pc_cpd = X_standardized_cpd[:, final_col_indices_cpd]

col_names_cpd = final_ids_cpd + ["CPD"]
X_pc_full_cpd = np.hstack([X_pc_cpd, Y_cpd.reshape(-1, 1)])

print("PC input shape:", X_pc_full_cpd.shape)

np.save(os.path.join(out_dir, "pc_input_cpd_final.npy"), X_pc_full_cpd)
with open(os.path.join(out_dir, "pc_col_names_cpd_final.json"), "w") as f:
    json.dump(col_names_cpd, f)
print("Saved.")

PC input shape: (1459, 4)
Saved.
